# E-Commerce Sales Performance & Customer Behavior Analysis
## Comprehensive End-to-End Data Analytics Portfolio Project
**Author:** Data Analytics Intern / AI & Data Science Candidate  
**Tools Used:** Python (Pandas, NumPy, Matplotlib, Seaborn), SQL (SQLite), Streamlit, Plotly  
**Domain:** Retail & E-Commerce Business Intelligence  

---

### 📋 Executive Business Context & Problem Statement
An established multi-category online retail platform experiences consistent daily user traffic but faces **stagnant revenue and decelerating quarterly profit margins**.

**Key Management Inquiries:**
1. Which product categories drive the highest revenue versus highest net profit?
2. Which product lines suffer from margin leakage due to excessive promotional discounting?
3. How do monthly and quarterly sales cycles fluctuate across seasonal peaks?
4. Who are the highest-value customer cohorts, and what are their retention patterns?
5. What concrete, data-backed operational levers can realistically bridge the gap toward a **+15% improvement in profit margin**?


## 1. Environment Setup & Library Imports

In [1]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings('ignore')

# Set visualization aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

print("Environment initialized successfully.")

## 2. Data Ingestion & Initial Health Audit
We ingest the raw sales dataset and perform an initial structural audit to identify missing records, duplicates, and anomalies.

In [2]:
# Load raw transactional dataset
raw_path = '../data/raw/ecommerce_sales_raw.csv' if os.path.exists('../data/raw/ecommerce_sales_raw.csv') else 'data/raw/ecommerce_sales_raw.csv'
df_raw = pd.read_csv(raw_path)

print(f"Raw Dataset Dimensions: {df_raw.shape[0]:,} Rows | {df_raw.shape[1]} Columns")
df_raw.head()

In [3]:
# Data quality inspection
print("--- Data Types & Non-Null Counts ---")
print(df_raw.info())

print("\n--- Missing Values Audit ---")
missing = df_raw.isnull().sum()
print(missing[missing > 0])

print(f"\n--- Duplicate Records Audit ---\nExact Duplicate Rows: {df_raw.duplicated().sum()}")

## 3. Systematic Data Cleaning & Transformation
Every cleaning step is rigorously executed and audited:
1. Deduplication
2. Text normalization and trimming
3. Domain-guided missing value imputation
4. Outlier & negative quantity elimination
5. Standardized formula recalculation for financial integrity

In [4]:
# 1. Deduplicate
df_clean = df_raw.drop_duplicates().copy()

# 2. Standardize column names
df_clean.columns = [c.strip().replace(' ', '_') for c in df_clean.columns]

# 3. Standardize text strings
cat_cols = ["Product_Category", "Sub_Category", "Product_Name", "Customer_Location", "Region", "Payment_Method", "Order_Status"]
for col in cat_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip().str.title()
        df_clean[col] = df_clean[col].replace('Nan', np.nan)

# 4. Missing value imputation
df_clean["Discount_Percentage"] = df_clean["Discount_Percentage"].fillna(0.0)
df_clean["Order_Status"] = df_clean["Order_Status"].fillna("Delivered")
df_clean["Payment_Method"] = df_clean["Payment_Method"].fillna("Credit Card")
loc_map = df_clean.dropna(subset=["Customer_Location"]).groupby("Customer_ID")["Customer_Location"].first().to_dict()
df_clean["Customer_Location"] = df_clean["Customer_Location"].fillna(df_clean["Customer_ID"].map(loc_map)).fillna("Chicago")

# 5. Discount range correction (>1.0 indicates whole percentage)
disc_mask = df_clean["Discount_Percentage"] > 1.0
df_clean.loc[disc_mask, "Discount_Percentage"] = df_clean.loc[disc_mask, "Discount_Percentage"] / 100.0

# 6. Remove invalid quantities
df_clean = df_clean[(df_clean["Quantity"] > 0) & (df_clean["Quantity"] <= 50)].copy()

# 7. Convert Order_Date
df_clean["Order_Date"] = pd.to_datetime(df_clean["Order_Date"])

# 8. Financial formula recalculation
cost_defaults = {"Electronics": 0.52, "Home & Kitchen": 0.55, "Fashion & Apparel": 0.38, "Beauty & Personal Care": 0.32, "Sports & Outdoors": 0.46}
df_clean["Sales_Amount"] = (df_clean["Quantity"] * df_clean["Unit_Price"] * (1 - df_clean["Discount_Percentage"])).round(2)

invalid_cost = (df_clean["Cost_Amount"] <= 0) | (df_clean["Cost_Amount"] > df_clean["Quantity"] * df_clean["Unit_Price"] * 1.5)
for cat, ratio in cost_defaults.items():
    m = invalid_cost & (df_clean["Product_Category"] == cat)
    df_clean.loc[m, "Cost_Amount"] = (df_clean.loc[m, "Quantity"] * df_clean.loc[m, "Unit_Price"] * ratio).round(2)

df_clean["Profit"] = (df_clean["Sales_Amount"] - df_clean["Cost_Amount"]).round(2)
df_clean["Profit_Margin"] = ((df_clean["Profit"] / df_clean["Sales_Amount"]) * 100).round(2)

print(f"Cleaning Complete: {len(df_clean):,} validated records retained.")

## 4. Feature Engineering & Customer Profiling
We extract temporal attributes and build customer-level metrics for behavior segmentation.

In [5]:
# Temporal feature extraction
df_clean["Year"] = df_clean["Order_Date"].dt.year
df_clean["Month"] = df_clean["Order_Date"].dt.month
df_clean["Month_Name"] = df_clean["Order_Date"].dt.strftime("%b")
df_clean["Quarter"] = "Q" + df_clean["Order_Date"].dt.quarter.astype(str)
df_clean["Year_Month"] = df_clean["Order_Date"].dt.strftime("%Y-%m")
df_clean["Day_of_Week"] = df_clean["Order_Date"].dt.dayofweek
df_clean["Weekday_Name"] = df_clean["Order_Date"].dt.strftime("%A")

# Discount Band assignment
def get_band(d):
    if d == 0: return "0% (No Discount)"
    elif d <= 0.10: return "1% - 10%"
    elif d <= 0.20: return "11% - 20%"
    else: return "> 20%"
df_clean["Discount_Band"] = df_clean["Discount_Percentage"].apply(get_band)

# Customer-level aggregation & RFM Quantile Segmentation
cust_summary = df_clean.groupby("Customer_ID").agg(
    Total_Orders=("Order_ID", "count"),
    Total_Revenue=("Sales_Amount", "sum"),
    Total_Profit=("Profit", "sum"),
    Average_Order_Value=("Sales_Amount", "mean"),
    Average_Discount=("Discount_Percentage", "mean"),
    First_Order_Date=("Order_Date", "min"),
    Last_Order_Date=("Order_Date", "max"),
    Customer_Location=("Customer_Location", "first"),
    Region=("Region", "first")
).reset_index()

q80 = cust_summary["Total_Revenue"].quantile(0.80)
q30 = cust_summary["Total_Revenue"].quantile(0.30)

def segment_cust(rev):
    if rev >= q80: return "High Value"
    elif rev >= q30: return "Medium Value"
    else: return "Low Value"

cust_summary["Customer_Segment"] = cust_summary["Total_Revenue"].apply(segment_cust)
cust_summary["Customer_Type"] = cust_summary["Total_Orders"].apply(lambda x: "Repeat Customer" if x > 1 else "One-Time Customer")

df_clean["Customer_Segment"] = df_clean["Customer_ID"].map(cust_summary.set_index("Customer_ID")["Customer_Segment"])
print(f"Engineered Features and segmented {len(cust_summary):,} unique customers.")

## 5. SQL Analytics Database Integration
We ingest the clean dataset into an in-memory / file SQLite database and execute production-grade analytical SQL queries.

In [6]:
conn = sqlite3.connect(':memory:')
df_clean.to_sql('sales_transactions', conn, index=False)
cust_summary.to_sql('customer_profiles', conn, index=False)

# Query: Core Executive KPIs
kpi_sql = """
SELECT 
    ROUND(SUM(Sales_Amount), 2) AS Total_Revenue,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(SUM(Profit) / SUM(Sales_Amount) * 100, 2) AS Overall_Profit_Margin_Pct,
    COUNT(DISTINCT Order_ID) AS Total_Orders,
    COUNT(DISTINCT Customer_ID) AS Total_Customers,
    ROUND(AVG(Sales_Amount), 2) AS Avg_Order_Value
FROM sales_transactions;
"""
pd.read_sql_query(kpi_sql, conn)

In [7]:
# Query: Category Financial Performance Breakdown
cat_sql = """
SELECT 
    Product_Category,
    COUNT(DISTINCT Order_ID) AS Order_Count,
    SUM(Quantity) AS Total_Units,
    ROUND(SUM(Sales_Amount), 2) AS Total_Revenue,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(SUM(Profit) / SUM(Sales_Amount) * 100, 2) AS Margin_Pct
FROM sales_transactions
GROUP BY Product_Category
ORDER BY Total_Revenue DESC;
"""
pd.read_sql_query(cat_sql, conn)

## 6. Exploratory Visual Analytics
Generating executive-ready charts across Revenue Trends, Category Performance, Customer Segments, and Pareto distributions.

In [8]:
# Visualization: Monthly Trends
monthly_agg = df_clean.groupby("Year_Month").agg({"Sales_Amount": "sum", "Profit": "sum"}).reset_index()

plt.figure(figsize=(11, 5))
plt.plot(monthly_agg["Year_Month"], monthly_agg["Sales_Amount"], marker='o', color='#1E88E5', linewidth=2.5, label='Gross Revenue ($)')
plt.plot(monthly_agg["Year_Month"], monthly_agg["Profit"], marker='s', color='#2E7D32', linewidth=2.5, label='Gross Profit ($)')
plt.title("Monthly Sales & Profit Performance (2025)", fontsize=13, fontweight='bold', pad=12)
plt.xlabel("Year-Month", fontsize=10)
plt.ylabel("Amount ($)", fontsize=10)
plt.xticks(rotation=45)
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
plt.legend()
plt.tight_layout()
plt.show()

In [9]:
# Visualization: 80/20 Pareto Profit Contribution by Category
cat_pareto = df_clean.groupby("Product_Category")["Profit"].sum().reset_index().sort_values("Profit", ascending=False)
cat_pareto["Cum_Profit_Pct"] = (cat_pareto["Profit"].cumsum() / cat_pareto["Profit"].sum()) * 100

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.bar(cat_pareto["Product_Category"], cat_pareto["Profit"], color='#1565C0', alpha=0.85)
ax2.plot(cat_pareto["Product_Category"], cat_pareto["Cum_Profit_Pct"], color='#D32F2F', marker='D', linewidth=2.5)
ax2.axhline(80, color='grey', linestyle='--', linewidth=1.2)
ax1.set_ylabel("Profit ($)", color='#1565C0')
ax2.set_ylabel("Cumulative Profit (%)", color='#D32F2F')
ax1.set_title("Pareto Profit Analysis by Category", fontsize=13, fontweight='bold')
plt.xticks(rotation=20, ha='right')
plt.show()

## 7. Target Margin Scenario Modeling (+15% Objective)
Evaluating feasibility of reaching the +15% quarterly profit margin objective through discount optimization and supplier renegotiation.

In [10]:
curr_margin = (df_clean["Profit"].sum() / df_clean["Sales_Amount"].sum()) * 100
target_margin = curr_margin * 1.15

print(f"Current Baseline Profit Margin: {curr_margin:.2f}%")
print(f"Target Profit Margin (+15% Relative Growth): {target_margin:.2f}%")
print(f"Required Margin Improvement Gap: {target_margin - curr_margin:.2f} percentage points\n")

# Scenario simulation
def simulate_levers(discount_cut_pct=2.0, cogs_reduction_pct=2.5):
    base_rev = df_clean["Sales_Amount"].sum()
    base_prof = df_clean["Profit"].sum()
    base_cogs = base_rev - base_prof
    
    sim_rev = base_rev * (1 + (discount_cut_pct / 100.0) * 0.8)
    sim_cogs = base_cogs * (1 - (cogs_reduction_pct / 100.0))
    sim_prof = sim_rev - sim_cogs
    sim_margin = (sim_prof / sim_rev) * 100
    
    return sim_rev, sim_prof, sim_margin

s_rev, s_prof, s_margin = simulate_levers(discount_cut_pct=2.0, cogs_reduction_pct=2.5)
print("--- Scenario Model Results ---")
print(f"Simulated Revenue: ${s_rev:,.2f}")
print(f"Simulated Gross Profit: ${s_prof:,.2f}")
print(f"Simulated Margin: {s_margin:.2f}%")
print(f"Target Achieved: {s_margin >= target_margin}")